## Download the exercise data
Run the next cell once before starting the exercise. It downloads and extracts this notebook’s data into `~/kenya2026`. Set `KENYA2026_WORK_DIR` first if you prefer another location.

In [ ]:
from pathlib import Path
import os
import subprocess

exercise = "day2_morning_sfs"
base_url = "https://popgen.dk/albrecht/course/kenya2026/data"
work_dir = Path(os.environ.get("KENYA2026_WORK_DIR", Path.home() / "kenya2026")).expanduser()
exercise_dir = work_dir / exercise
archive = work_dir / f"{exercise}.zip"
work_dir.mkdir(parents=True, exist_ok=True)

if not archive.exists():
    subprocess.run(["wget", "-c", f"{base_url}/{exercise}.zip", "-O", str(archive)], check=True)
if not exercise_dir.exists():
    subprocess.run(["unzip", "-q", str(archive), "-d", str(work_dir)], check=True)

os.chdir(exercise_dir)
print(f"Working directory: {Path.cwd()}")

### Software requirements
The setup cell above downloads **only the exercise data**. It does not install software. Before running the rest of this notebook, install the command-line programs and the Python or R packages that are imported or called in the exercises. If you see an error such as `command not found`, `ModuleNotFoundError`, or `there is no package called ...`, install the named dependency or ask an instructor for help.


# Site Frequency Spectrum (SFS) 


## Objectives

1. What is the SFS?
2. What is the neutral expectation for the SFS?
3. How to calculate the SFS & interpret the results


## 1. What is a site frequency spectrum?

For a sample of `n` haploid chromosomes, the **site frequency spectrum** summarizes how many segregating sites have been observed `i` times in the sample; ie the frequency of SNPs in our sample.

For an **unfolded SFS**, the bins are:

- `i = 1`: singleton variants
- `i = 2`: doubletons
- ...
- `i = n-1`: variants at the highest non-fixed frequency


For a **folded SFS**, ancestral and derived states are not known. Counts are folded around the midpoint, so frequency classes `i` and `n-i` are combined.

> **Important:** The SFS is a summary of genome-wide data, not the direct demographic history



## 2. Neutral expectation

Suppose we sample 20 haploid chromosomes. 
Under the standard neutral model, we expect number of sites with derived allele of frequency `i` to be proportional to `1/i`.

### Questions (1)

1. What is the x axis of the SFS?
2. What is the y axis of the SFS?
3. Which should be more common: singletons or variants with frequency 10?
4. What happens to the expected count as allele frequency increases?
5. What change to neutral expectation do you expect to see in population expansion?



In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# sample size 20 haploid chromosomes
n = 20
# possible segregating sites, 1,2..19
i = np.arange(1, n)

# expect sfs ∝ 1/i
# count 1 = 1/1, count 2 = 1/2 etc
neutral = 1 / i
# normalise by sum to get the relative frequencies (so freq classes add to 1)
neutral = neutral / neutral.sum()

# collect in df
sfs = pd.DataFrame({
    "derived_count": i,
    "relative_expected_frequency": neutral
})

# look at df
sfs.head()


In [ ]:
# barplot
plt.figure(figsize=(8, 4))
plt.bar(sfs["derived_count"], sfs["relative_expected_frequency"])
# add labels
plt.xlabel("Derived allele count")
# relative number of sites
plt.ylabel("Relative frequency")
# add title
plt.title("Neutral SFS expectation")
plt.show()


### Questions (2)

4. Under neutrality why do we see more rare variants than common ones?

A. Because mutations are unable to reach high frequency.  
B. Because neutral coalescent trees have longer branch lengths near the tips allowing rare mutations to occur.  
C. Because selection removes all common variants.  
D. Because sequencing errors always create singletons.


5. If a site has derived allele count `1` in a sample of 100 haploid chromosomes, it is:

A. A fixed difference  
B. A singleton  
C. A doubleton  
D. A common variant

## 3. Calculate the SFS from simulated data

There are 3 simulated datasets for you to read in, calculate the SFS for them & determine which demography produced them

In [ ]:
library(vcfR)

# calc sfs
calc_sfs <- function(gt) {
# columns = individuals
    n <- ncol(gt)
# rows = snps
    m <- nrow(gt)   
# calculate row sums (of the snps) 
    allele_counts <- rowSums(gt)
# normalise
    table(allele_counts) / m}

# vcf to sfs
one_scenario<-function(inp){
vcf <- read.vcfR(paste0("",inp,".vcf.gz"))
ingt <- extract.gt(vcf, element = "GT")

ingt[which(ingt=="0|0")]<-0
ingt[which(ingt=="0|1")]<-1
ingt[which(ingt=="1|0")]<-1
ingt[which(ingt=="1|1")]<-2

ingt <- apply(ingt, 2, as.numeric)

a_sfs<-calc_sfs(ingt)
return(a_sfs)}

scena<-one_scenario("simld_a_for_sfs")
scenb<-one_scenario("simld_b_for_sfs")
scenc<-one_scenario("simld_c_for_sfs")


In [ ]:
# make the plots
par(mfrow = c(1, 3))

barplot(scena,col = "darkseagreen" ,xlab = "Derived allele count",ylab = "Proportion of segregating sites",ylim = c(0, max(scena, scenb, scenc)),border = NA)

barplot(scenb,col = "plum",border = NA, xlab = "Derived allele count",ylab = "Proportion of segregating sites",ylim = c(0, max(scena, scenb, scenc)))

barplot(scenc,col = "blue",border = NA, xlab = "Derived allele count",ylab = "Proportion of segregating sites",ylim = c(0, max(scena, scenb, scenc)))

### Questions (3)

1. Which scenario do you think was used to generate each of a,b,c?
2. Describe the differences in the SFS shape for each curve
3. Does the curve for c look unusual? why may this be?

## 4. Calculate the SFS from wildebeest data

Calculate the SFS for wildebeest data & interpret results

In [ ]:

library(vcfR)

# folded sfs function
calc_folded<-function(ingen){
nchr <- 2*ncol(ingen)       
# count alt alleles per site
alt_count <- rowSums(ingen, na.rm = TRUE)
# folding, by taking smaller of the alt or reference allele count
minor_count <- pmin(alt_count, nchr - alt_count)
# count how many sites per category
folded_sfs <- table(minor_count)
return(folded_sfs)}

# read in vcf, extract gt matrix, convert to matrix of 0,1,2, use this matrix to calc folded sfs 
vcf_to_foldedsfs<-function(inp){
vcf <- read.vcfR(paste0("",inp,"_chr1.vcf.gz"))
ingt <- extract.gt(vcf, element = "GT")
# remove rows with nas
ingt <- ingt[complete.cases(ingt), ]

ingt[which(ingt=="0/0")]<-0
ingt[which(ingt=="0/1")]<-1
ingt[which(ingt=="1/0")]<-1
ingt[which(ingt=="1/1")]<-2

ingt <- apply(ingt, 2, as.numeric)
emp1<-calc_folded(ingt)
return(emp1)}

# black wildebeest
blackwildebeest<-vcf_to_foldedsfs("blackwildebeest")


# blue wildebeest white beard 
bluewildebeest<-vcf_to_foldedsfs("bluewildebeest_whitebeard")

In [ ]:
# plot without the 0 sites
blackwildebeest1 <- blackwildebeest[names(blackwildebeest) != "0"]
bluewildebeest1 <- bluewildebeest[names(bluewildebeest) != "0"]
par(mfrow = c(1, 2))
barplot(blackwildebeest1, col = "black" ,xlab = "Derived allele count",ylab = "Counts of segregating sites",border = NA)
barplot(bluewildebeest1, col = "blue" ,xlab = "Derived allele count",ylab = "Counts of segregating sites",border = NA)

### Questions (4)

1. What is different about the SFS you calculated here for the wildebeest vs the simulated data above? & why do you think we did this differently?
2. What is the difference between the 2 plots?
3. Look at shape of SFS what can you conclude about these 2 populations?

In [ ]:
# plot. using the same y limits
par(mfrow = c(1, 2))
barplot(blackwildebeest1, col = "black" ,xlab = "Derived allele count",ylab = "Counts of segregating sites",border = NA, ylim=c(0,max(c(bluewildebeest1,blackwildebeest1))))
barplot(bluewildebeest1, col = "blue" ,xlab = "Derived allele count",ylab = "Counts of segregating sites",border = NA, ylim=c(0,max(c(bluewildebeest1,blackwildebeest1))))


### Questions (5)
What information is lost when an SFS is folded?

A. The total number of segregating sites
B. The distinction between low-frequency derived alleles and high-frequency derived alleles
C. The sample size
D. All information about genetic variation

### Questions (6)

1. What does SFS bin `i` represent?

A. Number of mutations per chromosome  
B. Number of segregating sites whose derived allele occurs `i` times  
C. Number of chromosomes sampled from population `i`  
D. Number of genes in population `i`


2. Under the standard neutral constant-size model, the expected SFS is approximately proportional to:

A. `i`  
B. `i²`  
C. `1/i`  
D. constant for all `i`


3. Why fold an SFS?

A. To increase sample size  
B. To handle uncertainty about the ancestral state  
C. To remove all sequencing errors  
D. To estimate recombination directly

